# sLDA Model Explained




# Data

CompIRT turns readable rows with string IDs into tensors with integer indexes.  
The indexes are pointers into learned parameter tables.

Example:

```python
player_ids = ["alice", "bob", "carol"]
data["player"] = tensor([0, 1, 2, 0])
```

means:

```text
row 0 -> alice
row 1 -> bob
row 2 -> carol
row 3 -> alice again
```

So:

```python
theta[data["player"]]
```

means:

```text
[theta[0], theta[1], theta[2], theta[0]]
```

In other words, it selects the correct player skill vector for every observation row.

The text after `#` in code is just a comment:

```python
theta[data["player"]]  # player skill per row
```

Only `theta[data["player"]]` is code.  
`# player skill per row` is a note explaining what the code returns.

---

## Stage 1: Tossup Model

```python
data = tossup_tensors(tossup_observations)
```

Each row = **one player in one game-tossup event**.

Example readable rows:

```text
row | player_id | tossup_id | event_id  | z        | t_star | T_j
0   | alice     | q1        | game7::q1 | CENSORED | 57     | 117
1   | bob       | q1        | game7::q1 | BUZZ     | 57     | 117
2   | carol     | q1        | game7::q1 | CENSORED | 57     | 117
3   | alice     | q2        | game8::q2 | NO-BUZZ  | 100    | 100
```

Lookup lists:

```python
player_ids = ["alice", "bob", "carol"]
tossup_ids = ["q1", "q2"]
event_ids = ["game7::q1", "game8::q2"]
```

Tensor version:

```python
data["player"] = tensor([0, 1, 2, 0])
data["tossup"] = tensor([0, 0, 0, 1])
data["event"]  = tensor([0, 0, 0, 1])
```

Main tossup tensors:

| Tensor | Meaning |
| --- | --- |
| `player` | player index per row |
| `tossup` | tossup index per row |
| `event` | game-tossup event index |
| `t_star` | observed stopping token position |
| `T_j` | total tossup tokens |
| `y` | correctness label, only meaningful for buzz rows |
| `is_buzz` | true for first-buzzer rows |
| `is_no_buzz` | true for dead tossup rows |

The tossup model learns:

| Parameter | Meaning |
| --- | --- |
| `theta` | player skill |
| `c` | player buzz threshold |
| `alpha` | tossup discrimination |
| `beta` | tossup difficulty |
| `gamma` | difficulty curve sharpness |
| `mu` | difficulty curve midpoint |

Indexes are used like this:

```python
theta[data["player"]]  # player skill per row
c[data["player"]]      # buzz threshold per row

alpha[data["tossup"]]  # tossup discrimination per row
beta[data["tossup"]]   # tossup difficulty per row
gamma[data["tossup"]]  # curve sharpness per row
mu[data["tossup"]]     # curve midpoint per row
```

“Per row” means each observation row gets the correct player and tossup parameters.

Example:

```text
row 0 uses theta[alice], c[alice], alpha[q1], beta[q1], gamma[q1], mu[q1]
row 1 uses theta[bob],   c[bob],   alpha[q1], beta[q1], gamma[q1], mu[q1]
row 3 uses theta[alice], c[alice], alpha[q2], beta[q2], gamma[q2], mu[q2]
```

Tossup model question:

```text
For this player, on this tossup, at this token position,
did they buzz first, answer correctly, or survive without buzzing?
```

---

## Stage 2: Bonus Model

```python
data = bonus_tensors(bonus_observations, player_ids)
```

Each row = **one team attempting one bonus part**.

Example readable rows:

```text
row | team_id | player_ids   | bonus_part_id | correct
0   | team-a  | [alice,bob]  | b1::part-1    | 1
1   | team-a  | [alice,bob]  | b1::part-2    | 0
2   | team-b  | [carol]      | b1::part-1    | 1
```

Tensor version:

```python
data["roster"] = tensor([
  [0, 1],
  [0, 1],
  [2, 0],
])

data["roster_mask"] = tensor([
  [True,  True],
  [True,  True],
  [True,  False],
])

data["part"] = tensor([0, 1, 0])
data["y"] = tensor([1.0, 0.0, 1.0])
```

Here:

```text
[0, 1] -> alice and bob
[2, 0] with mask [True, False] -> carol only; second slot is padding
```

Main bonus tensors:

| Tensor | Meaning |
| --- | --- |
| `roster` | player indexes for each team |
| `roster_mask` | true for real players, false for padding |
| `part` | bonus-part index |
| `y` | 1 if correct, 0 if incorrect |
| `part_ids` | lookup list for bonus part IDs |

The bonus model reuses `theta` from Stage 1 and learns:

| Parameter | Meaning |
| --- | --- |
| `alpha` | bonus-part discrimination |
| `beta` | bonus-part difficulty |
| `tau` | team aggregation temperature |
| `psi` | optional player synergy vectors |

Indexes are used like this:

```python
theta[data["roster"]]  # player skills for all roster members
alpha[data["part"]]    # bonus-part discrimination per row
beta[data["part"]]     # bonus-part difficulty per row
```

Bonus model question:

```text
Given this roster and this bonus part,
what is the probability the team gets it correct?
```

---

## Memory Hook

```text
player index -> row in theta and c
tossup index -> row in tossup alpha/beta/gamma/mu
event index -> groups rows from the same game-tossup
part index -> row in bonus alpha/beta
roster index -> multiple player indexes for one team
```

Stage 1 learns player skill `theta` from tossup behavior.  
Stage 2 reuses `theta` with team rosters to predict bonus part correctness.

# CompIRT Base 

## Top Level 